# CareLoop Clinics — No-Show Guard

## Part 3: Machine Learning

**Project Objective:** Build and evaluate machine-learning models to predict patient appointment no-shows.

**Dataset:** `appointments_clean.csv`

**Target Variable:** `no_show`

**Models Used:**
- Logistic Regression
- Decision Tree
- Random Forest

**Evaluation:** Accuracy, Precision, Recall, F1-score and ROC-AUC.

## Task 3.1 — Features, Split and Baseline

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

df_ml = pd.read_csv("appointments_clean.csv")

print("Dataset shape:", df_ml.shape)

Dataset shape: (990, 16)


### 3.1.1 — Feature Selection

Selecting the 11 input features used to predict appointment no-shows. The target variable is `no_show`.

In [ ]:
X = df_ml[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

y = df_ml["no_show"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (990, 11)
Target shape: (990,)


### 3.1.2 — Train-Test Split and Baseline Accuracy

Splitting the dataset into 80% training data and 20% testing data using `random_state=42`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("No-shows in test set:", y_test.sum())

baseline_accuracy = (y_test == 0).mean()
print("Baseline accuracy:", round(baseline_accuracy, 4))

Training rows: 792
Testing rows: 198
No-shows in test set: 49
Baseline accuracy: 0.7525


## Task 3.2 — Logistic Regression

Building a Logistic Regression model to predict appointment no-shows and evaluating its performance using classification metrics.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

print("Model training completed.")

Model training completed.


In [ ]:
y_pred = model.predict(X_test_scaled)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1 Score:", round(f1_score(y_test, y_pred), 4))

Confusion Matrix:
[[139  10]
 [ 32  17]]

Accuracy: 0.7879
Precision: 0.6296
Recall: 0.3469
F1 Score: 0.4474


In [ ]:
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

ROC-AUC: 0.7889


In [ ]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

print(coefficients.sort_values(
    by="Coefficient",
    ascending=False
).to_string(index=False))

                Feature  Coefficient
         prior_no_shows     0.649612
              lead_days     0.614471
                fee_inr     0.452761
            distance_km     0.253448
  booking_channel_Phone     0.215881
visit_type_Consultation     0.015658
   visit_type_Follow-up    -0.015658
    booking_channel_App    -0.046804
booking_channel_Walk-In    -0.186364
                    age    -0.253880
          reminder_sent    -0.580669


In [ ]:
reminder_index = list(X.columns).index("reminder_sent")

reminder_coefficient = (
    model.coef_[0][reminder_index] /
    scaler.scale_[reminder_index]
)

odds_ratio = np.exp(reminder_coefficient)

print("Reminder coefficient:", round(reminder_coefficient, 4))
print("Odds ratio:", round(odds_ratio, 4))

Reminder coefficient: -1.1731
Odds ratio: 0.3094


### Interpretation

The Logistic Regression model achieved an accuracy of 78.79% and a ROC-AUC score of 0.7889.

The reminder_sent coefficient is negative (-1.1731), with an odds ratio of 0.3094. This indicates that appointments with reminders had approximately 69% lower odds of no-show in the fitted model, holding other features constant.

This is an association observed in the model and does not establish causation.

## Task 3.3 — Decision Tree and Overfitting

Training Decision Tree models with different depths to understand overfitting and compare their performance on training and testing data.

In [ ]:
tree_full = DecisionTreeClassifier(random_state=42)

tree_full.fit(X_train, y_train)

print("Full Tree Training Accuracy:",
      round(tree_full.score(X_train, y_train), 4))

print("Full Tree Testing Accuracy:",
      round(tree_full.score(X_test, y_test), 4))

Full Tree Training Accuracy: 1.0
Full Tree Testing Accuracy: 0.6717


In [ ]:
tree_limited = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

tree_limited.fit(X_train, y_train)

print("Limited Tree Training Accuracy:",
      round(tree_limited.score(X_train, y_train), 4))

print("Limited Tree Testing Accuracy:",
      round(tree_limited.score(X_test, y_test), 4))

Limited Tree Training Accuracy: 0.822
Limited Tree Testing Accuracy: 0.7778


### Interpretation — Overfitting

The full Decision Tree overfits the training data because it achieves 100% training accuracy but only 67.17% testing accuracy.

Limiting the tree depth to 3 reduces overfitting and improves testing accuracy to 77.78%. This shows that a simpler model can generalise better to unseen data.

## Task 3.4 — Random Forest and Model Comparison

Training a Random Forest classifier and comparing its performance with Logistic Regression and Decision Tree models.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [ ]:
rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:",
      round(accuracy_score(y_test, rf_pred), 4))

print("Precision:",
      round(precision_score(y_test, rf_pred), 4))

print("Recall:",
      round(recall_score(y_test, rf_pred), 4))

print("F1 Score:",
      round(f1_score(y_test, rf_pred), 4))

print("ROC-AUC:",
      round(roc_auc_score(
          y_test,
          rf_model.predict_proba(X_test)[:, 1]
      ), 4))

Random Forest Accuracy: 0.7475
Precision: 0.4839
Recall: 0.3061
F1 Score: 0.375
ROC-AUC: 0.7057


In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.head(3))

       Feature  Importance
3    lead_days    0.219905
0          age    0.196788
1  distance_km    0.194739


### Random Forest — Feature Importance

The three most important features in the Random Forest model are lead_days, age and distance_km.

Among these, lead_days has the highest feature importance (0.219905), followed by age (0.196788) and distance_km (0.194739).

This indicates that these features contributed most to the Random Forest's predictions. Feature importance does not imply causation.

In [ ]:
models = {
    "Logistic Regression": (
        model, X_test_scaled
    ),
    "Full Decision Tree": (
        tree_full, X_test
    ),
    "Limited Decision Tree": (
        tree_limited, X_test
    ),
    "Random Forest": (
        rf_model, X_test
    )
}

comparison = []

for name, (clf, test_data) in models.items():
    predictions = clf.predict(test_data)
    probabilities = clf.predict_proba(test_data)[:, 1]

    comparison.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

comparison_df = pd.DataFrame(comparison)

print(comparison_df.round(4).to_string(index=False))

                Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC
  Logistic Regression    0.7879     0.6296  0.3469    0.4474   0.7889
   Full Decision Tree    0.6717     0.3519  0.3878    0.3689   0.5764
Limited Decision Tree    0.7778     0.6471  0.2245    0.3333   0.6997
        Random Forest    0.7475     0.4839  0.3061    0.3750   0.7057


### Model Comparison — Interpretation

Logistic Regression performs best overall, achieving the highest accuracy (78.79%), F1-score (0.4474) and ROC-AUC (0.7889).

The Limited Decision Tree achieves the highest precision (0.6471), while the Full Decision Tree achieves the highest recall (0.3878).

Based on the overall comparison, Logistic Regression is the strongest model among the four evaluated models.

### 3.5.1 — Random Forest Hyperparameter Tuning

Using GridSearchCV with 5-fold cross-validation to find the best Random Forest settings based on ROC-AUC.

In [ ]:
from sklearn.model_selection import (
    GridSearchCV,
    cross_val_score,
    cross_val_predict
)

from sklearn.pipeline import Pipeline

In [ ]:
rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid={
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 8]
    },
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Best parameters:", rf_grid.best_params_)
print("Best CV ROC-AUC:", round(rf_grid.best_score_, 4))

Best parameters: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.7673


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# Logistic Regression with scaling inside each CV fold
log_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

log_cv = cross_val_score(
    log_pipeline, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

# Depth-3 Decision Tree
tree_cv = cross_val_score(
    DecisionTreeClassifier(max_depth=3, random_state=42),
    X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

# Random Forest from Task 3.4
rf_cv = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

print("Logistic Regression CV ROC-AUC:", round(log_cv, 4))
print("Depth-3 Decision Tree CV ROC-AUC:", round(tree_cv, 4))
print("Random Forest CV ROC-AUC:", round(rf_cv, 4))
print("Tuned Random Forest CV ROC-AUC:", round(rf_grid.best_score_, 4))

Logistic Regression CV ROC-AUC: 0.7737
Depth-3 Decision Tree CV ROC-AUC: 0.6784
Random Forest CV ROC-AUC: 0.7466
Tuned Random Forest CV ROC-AUC: 0.7673


In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

oof_probabilities = cross_val_predict(
    log_pipeline,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

for threshold in [0.5, 0.35]:
    predictions = (oof_probabilities >= threshold).astype(int)

    print("\nThreshold:", threshold)
    print("Precision:", round(precision_score(y_train, predictions), 4))
    print("Recall:", round(recall_score(y_train, predictions), 4))
    print("F1 Score:", round(f1_score(y_train, predictions), 4))
    print("Number flagged:", predictions.sum())


Threshold: 0.5
Precision: 0.5753
Recall: 0.2485
F1 Score: 0.3471
Number flagged: 73

Threshold: 0.35
Precision: 0.513
Recall: 0.4675
F1 Score: 0.4892
Number flagged: 154


### 3.5.2 — Cross-Validation Model Comparison

Comparing Logistic Regression, Decision Tree and tuned Random Forest using 5-fold cross-validation ROC-AUC on the training data.

In [ ]:
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

models_cv = {
    "Logistic Regression": lr_pipeline,
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3, random_state=42
    ),
    "Tuned Random Forest": rf_grid.best_estimator_
}

for name, estimator in models_cv.items():
    scores = cross_val_score(
        estimator,
        X_train,
        y_train,
        cv=5,
        scoring="roc_auc"
    )
    print(name, ":", round(scores.mean(), 4))

Logistic Regression : 0.7737
Decision Tree : 0.6784
Tuned Random Forest : 0.7721


### 3.5.3 — Threshold Selection

Comparing classification thresholds of 0.50 and 0.35 using out-of-fold predictions on the training data. The test set is not used for threshold selection.

In [ ]:
final_model = log_pipeline

final_model.fit(X_train, y_train)

print("Final Logistic Regression model fitted on training data.")

Final Logistic Regression model fitted on training data.


In [ ]:
oof_probabilities = cross_val_predict(
    lr_pipeline,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

for threshold in [0.50, 0.35]:
    oof_predictions = (
        oof_probabilities >= threshold
    ).astype(int)

    score = f1_score(y_train, oof_predictions)

    print(
        "Threshold:", threshold,
        "| F1-score:", round(score, 4)
    )

Threshold: 0.5 | F1-score: 0.3471
Threshold: 0.35 | F1-score: 0.4892


### 3.5.4 — Final Model Evaluation

The selected Logistic Regression model is trained on the complete training dataset. Its final performance is evaluated on the test dataset using the selected classification threshold of 0.35.

In [ ]:
test_probabilities = final_model.predict_proba(X_test)[:, 1]

test_predictions = (
    test_probabilities >= 0.35
).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

print("Accuracy:", round(
    accuracy_score(y_test, test_predictions), 4
))

print("Precision:", round(
    precision_score(y_test, test_predictions), 4
))

print("Recall:", round(
    recall_score(y_test, test_predictions), 4
))

print("F1 Score:", round(
    f1_score(y_test, test_predictions), 4
))

print("ROC-AUC:", round(
    roc_auc_score(y_test, test_probabilities), 4
))

Confusion Matrix:
[[128  21]
 [ 21  28]]
Accuracy: 0.7879
Precision: 0.5714
Recall: 0.5714
F1 Score: 0.5714
ROC-AUC: 0.7889


In [ ]:
final_risk_scores = pd.DataFrame({
    "appointment_id": df_ml.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": test_probabilities,
    "flag": test_predictions
})

final_risk_scores.to_csv("test_risk_scores.csv", index=False)

print("CSV shape:", final_risk_scores.shape)
print("Columns:", final_risk_scores.columns.tolist())
display(final_risk_scores.head())

CSV shape: (198, 4)
Columns: ['appointment_id', 'no_show', 'risk_prob', 'flag']


,appointment_id,no_show,risk_prob,flag
0,A0217,0,0.021565,0
1,A0336,0,0.150917,0
2,A0508,0,0.518692,1
3,A0314,0,0.191300,0
4,A0089,0,0.167718,0


In [ ]:
from google.colab import files
files.download("test_risk_scores.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Train the selected model on all training data
lr_pipeline.fit(X_train, y_train)

# Final test predictions
test_probabilities = lr_pipeline.predict_proba(X_test)[:, 1]

test_predictions = (
    test_probabilities >= 0.35
).astype(int)

# Final evaluation
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

print("\nAccuracy:",
      round(accuracy_score(y_test, test_predictions), 4))

print("Precision:",
      round(precision_score(y_test, test_predictions), 4))

print("Recall:",
      round(recall_score(y_test, test_predictions), 4))

print("F1 Score:",
      round(f1_score(y_test, test_predictions), 4))

print("ROC-AUC:",
      round(roc_auc_score(y_test, test_probabilities), 4))

Confusion Matrix:
[[128  21]
 [ 21  28]]

Accuracy: 0.7879
Precision: 0.5714
Recall: 0.5714
F1 Score: 0.5714
ROC-AUC: 0.7889


In [ ]:
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

models_cv = {
    "Logistic Regression": logistic_pipeline,
    "Limited Decision Tree": DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ),
    "Tuned Random Forest": rf_grid.best_estimator_
}

cv_results = {}

for name, clf in models_cv.items():
    scores = cross_val_score(
        clf,
        X_train,
        y_train,
        cv=5,
        scoring="roc_auc"
    )

    cv_results[name] = scores.mean()

print("Cross-validation ROC-AUC:")
for name, score in cv_results.items():
    print(name, round(score, 4))

Cross-validation ROC-AUC:
Logistic Regression 0.7737
Limited Decision Tree 0.6784
Tuned Random Forest 0.7721


### 3.5.2 — Threshold Selection

Comparing classification thresholds of 0.50 and 0.35 using five-fold cross-validation predictions on the training data. The F1-score is used to compare the thresholds.

In [ ]:
oof_probabilities = cross_val_predict(
    logistic_pipeline,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

for threshold in [0.50, 0.35]:
    oof_predictions = (
        oof_probabilities >= threshold
    ).astype(int)

    print("\nThreshold:", threshold)
    print("Precision:", round(
        precision_score(y_train, oof_predictions), 4
    ))
    print("Recall:", round(
        recall_score(y_train, oof_predictions), 4
    ))
    print("F1 Score:", round(
        f1_score(y_train, oof_predictions), 4
    ))


Threshold: 0.5
Precision: 0.5753
Recall: 0.2485
F1 Score: 0.3471

Threshold: 0.35
Precision: 0.513
Recall: 0.4675
F1 Score: 0.4892


### 3.5.3 — Final Model Training

The Logistic Regression model was selected based on cross-validation ROC-AUC. A classification threshold of 0.35 was selected based on the training data's cross-validation F1-score.

In [ ]:
final_model = logistic_pipeline

final_model.fit(X_train, y_train)

print("Final model training completed.")
print("Selected threshold: 0.35")

Final model training completed.
Selected threshold: 0.35


In [ ]:
test_probabilities = final_model.predict_proba(X_test)[:, 1]

test_predictions = (
    test_probabilities >= 0.35
).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

print("Accuracy:", round(
    accuracy_score(y_test, test_predictions), 4
))

print("Precision:", round(
    precision_score(y_test, test_predictions), 4
))

print("Recall:", round(
    recall_score(y_test, test_predictions), 4
))

print("F1 Score:", round(
    f1_score(y_test, test_predictions), 4
))

print("ROC-AUC:", round(
    roc_auc_score(y_test, test_probabilities), 4
))

Confusion Matrix:
[[128  21]
 [ 21  28]]
Accuracy: 0.7879
Precision: 0.5714
Recall: 0.5714
F1 Score: 0.5714
ROC-AUC: 0.7889


In [ ]:
risk_scores = pd.DataFrame({
    "appointment_id": df_ml.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": test_probabilities,
    "flag": (test_probabilities >= 0.35).astype(int)
})

risk_scores.to_csv("test_risk_scores.csv", index=False)

print(risk_scores.head())
print("Rows saved:", len(risk_scores))

  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0
Rows saved: 198


In [ ]:
print(risk_scores.shape)
print(risk_scores.columns.tolist())
print(risk_scores["flag"].value_counts())

(198, 4)
['appointment_id', 'no_show', 'risk_prob', 'flag']
flag
0    149
1     49
Name: count, dtype: int64
